# FightWatch — chuẩn bị nhãn từng đoạn trên CPU
Không cần GPU. Dùng AIRTLab/RLVS public đã có; không đọc/tải video iPhone. Chọn nguồn theo SHA và nhóm biểu diễn, không theo điểm model. Mọi nguồn bắt đầu ở PENDING; không tự biến nhãn cả video thành nhãn từng giây.
Ưu tiên VIA3 chính thức (Oxford), không tự viết lại timeline. Tự giải phóng runtime CPU sau khi chuẩn bị; Drive giữ nguyên. Watchdog 20 phút, có thể chạy lại để tiếp tục preview dang dở.

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path
import threading
import torch
from google.colab import drive, runtime
if torch.cuda.is_available():
    runtime.unassign()
    raise RuntimeError('Notebook này chỉ chuẩn bị dữ liệu bằng CPU. Chọn loại runtime CPU, không dùng GPU.')
if 'CPU_WATCHDOG' not in globals():
    CPU_WATCHDOG=threading.Timer(20*60,runtime.unassign)
    CPU_WATCHDOG.daemon=True
    CPU_WATCHDOG.start()
try:
    drive.mount('/content/drive',timeout_ms=180000)
    ROOT=Path('/content/drive/MyDrive/SENTINEL')
    REPO=ROOT/'repo/anomaly-behavior-detector-system'
    LOCAL=Path('/content/fightwatch-refinement-code')
    SOURCE=ROOT/'artifacts/fight_fusion_v1'
    REVIEW=ROOT/'artifacts/fight_temporal_review_v1'
    DATA=Path('/content/fightwatch-reviewed-public-data')
    subprocess.run(['git','-C',str(REPO),'restore','--','colab'],check=True,timeout=60)
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True,timeout=60)
    assert not dirty.strip(), dirty
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'],check=True,timeout=90)
    REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    shutil.copytree(REPO,LOCAL,dirs_exist_ok=True,ignore=shutil.ignore_patterns('.git','.venv','.env','logs','uploads','models','__pycache__','.pytest_cache','datasets','artifacts'))
    os.chdir(LOCAL)
    from scripts.colab_process import run_logged
    print('Code:',REVISION)
except BaseException:
    runtime.unassign()
    raise


In [ ]:
try:
    run_logged([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],REVIEW/'setup.log',600,revision=REVISION)
    pack=REVIEW/'review_pack'
    if not (pack/'review.json').exists():
        base=ROOT/'artifacts/fight_large_v1'
        assert (base/'manifest.json').exists(), 'Không tạo lại split; cần manifest public gốc'
        run_logged([sys.executable,'-m','scripts.fight_public_data','--archives',str(ROOT/'datasets/fight'),'--data',str(DATA),'--output',str(base)],REVIEW/'data.log',900,revision=REVISION)
        run_logged([sys.executable,'-m','scripts.fight_temporal_review','prepare','--manifest',str(SOURCE/'source_manifest.json'),'--data',str(DATA),'--output',str(pack)],REVIEW/'prepare.log',600,revision=REVISION)
    else:
        print('Giữ nguyên preview/nhãn đã tạo; chỉ cập nhật công cụ review.')
    run_logged([sys.executable,'-m','scripts.fight_temporal_review','refresh-ui','--output',str(pack)],REVIEW/'refresh.log',60,revision=REVISION)
    run_logged([sys.executable,'-m','scripts.fight_via','prepare','--pack',str(pack),'--with-app'],REVIEW/'via_setup.log',120,revision=REVISION)
    import hashlib
    files=sorted(path for path in pack.rglob('*') if path.is_file())
    signature=hashlib.sha256(json.dumps([(str(path.relative_to(pack)),hashlib.sha256(path.read_bytes()).hexdigest()) for path in files],sort_keys=True).encode()).hexdigest()
    archive=REVIEW/'temporal_review_pack.zip'
    marker=REVIEW/'package_signature.json'
    if not archive.exists() or not marker.exists() or json.loads(marker.read_text()).get('sha256')!=signature:
        partial=REVIEW/'temporal_review_pack.partial'
        shutil.make_archive(str(partial),'zip',pack)
        partial.with_suffix('.partial.zip').replace(archive)
        marker.write_text(json.dumps({'sha256':signature,'code_revision':REVISION},indent=2))
    print('Gói review public:',archive,flush=True)
    print('Giải nén ở local, chạy: python -m http.server 8002 --bind 127.0.0.1 --directory <thư mục giải nén>',flush=True)
    print('Mở http://127.0.0.1:8002/via_video_annotator.html, load via_project.json, gán mốc/action rồi save project VIA.',flush=True)
    print('Dùng scripts.fight_via import để chuyển JSON VIA; không tự duyệt. Xem docs/OPEN_SOURCE_COMPARISON.md.',flush=True)
    print('Đặt annotations.reviewed.json đã chuyển/duyệt tại:',REVIEW/'annotations.reviewed.json',flush=True)
    annotations=REVIEW/'annotations.reviewed.json'
    if annotations.exists():
        run_logged([sys.executable,'-m','scripts.fight_temporal_review','check','--manifest',str(SOURCE/'source_manifest.json'),'--annotations',str(annotations)],REVIEW/'readiness.log',60,revision=REVISION)
        print('Preflight nhãn đã qua. Chưa tự bật GPU; cần ngân sách mới trước training.',flush=True)
    else:
        print('CHỜ DUYỆT: chưa có nhãn thời gian được xác nhận; training vẫn bị chặn. Không bịa nhãn từ nhãn cả clip.',flush=True)
finally:
    print('Giải phóng runtime CPU; giữ gói review và log trên Drive.',flush=True)
    runtime.unassign()


## Sau khi duyệt
Gói test public đã được xem trước, không phải kiểm thử độc lập mới. AI hỗ trợ phải ghi ai_assisted. Gán nhãn xong, đưa annotations.reviewed.json vào thư mục in trên màn hình và chạy lại notebook CPU để kiểm preflight; preview đã có không cần giải nén lại. Chỉ mở fight_refinement_drive sau khi nhãn đạt điều kiện và đã duyệt ngân sách GPU mới.